In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# 1. Load Bronze Customers
# ============================================================

bronze_customers_df = spark.table(
    "customer360_dev.bronze.customers"
)


# ============================================================
# 2. Read All Incremental Customer Rows
# ============================================================

latest_run_row = (
    bronze_customers_df
    .filter(
        F.col("_load_type") == "INCREMENTAL"
    )
    .filter(
        F.col("_pipeline_run_id").isNotNull()
    )
    .groupBy("_pipeline_run_id")
    .agg(
        F.max("_ingestion_timestamp").alias(
            "latest_ingestion_timestamp"
        )
    )
    .orderBy(
        F.col("latest_ingestion_timestamp").desc()
    )
    .first()
)

run_id = (
    latest_run_row["_pipeline_run_id"]
    if latest_run_row
    else None
)

print("Processing Run ID:", run_id)

if run_id is not None:

    incremental_bronze_df = (
        spark.table(
        "customer360_dev.bronze.customers"
        )
        .filter(
        F.col("_pipeline_run_id") == run_id
        )
    )

else:

    incremental_bronze_df = bronze_customers_df.limit(0)

    print(
        "No incremental customer batch "
        "available."
    )

# ============================================================
# 3. Count Incremental Rows
# ============================================================

incremental_count = incremental_bronze_df.count()

print(
    f"Incremental customer rows: "
    f"{incremental_count:,}"
)


# ============================================================
# 4. Load Current Silver Customers
# ============================================================

silver_customers_df = spark.table(
    "customer360_dev.silver.customers"
)


# ============================================================
# 5. Clean and Cast Incremental Records
# ============================================================

incremental_clean_df = (
    incremental_bronze_df
    .withColumn(
        "first_name_clean",
        F.trim("first_name")
    )
    .withColumn(
        "last_name_clean",
        F.trim("last_name")
    )
    .withColumn(
        "email_clean",
        F.lower(F.trim("email"))
    )
    .withColumn(
        "phone_number_clean",
        F.trim("phone_number")
    )
    .withColumn(
        "date_of_birth_cast",
        F.coalesce(
            F.try_to_date("date_of_birth", "yyyy-MM-dd"),
            F.try_to_date("date_of_birth", "dd-MM-yyyy")
        )
    )
    .withColumn(
        "gender_clean",
        F.upper(F.trim("gender"))
    )
    .withColumn(
        "signup_date_cast",
        F.coalesce(
            F.try_to_date("signup_date", "yyyy-MM-dd"),
            F.try_to_date("signup_date", "dd-MM-yyyy")
        )
    )
    .withColumn(
        "customer_status_clean",
        F.upper(F.trim("customer_status"))
    )
    .withColumn(
        "loyalty_tier_clean",
        F.upper(F.trim("loyalty_tier"))
    )
    .withColumn(
        "preferred_language_clean",
        F.upper(F.trim("preferred_language"))
    )
    .withColumn(
        "created_at_cast",
        F.coalesce(
            F.try_to_timestamp(F.col("created_at"), F.lit("yyyy-MM-dd HH:mm:ss")),
            F.try_to_timestamp(F.col("created_at"), F.lit("dd-MM-yyyy HH:mm"))
        )
    )
    .withColumn(
        "updated_at_cast",
        F.coalesce(
            F.try_to_timestamp(F.col("updated_at"), F.lit("yyyy-MM-dd HH:mm:ss")),
            F.try_to_timestamp(F.col("updated_at"), F.lit("dd-MM-yyyy HH:mm"))
        )
    )
)


# ============================================================
# 6. Deduplicate Incremental Customers
# ============================================================

incremental_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        F.col("updated_at_cast").desc_nulls_last(),
        F.col("_ingestion_timestamp").desc()
    )
)

incremental_dedup_df = (
    incremental_clean_df
    .withColumn(
        "_row_number",
        F.row_number().over(
            incremental_window
        )
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)


# ============================================================
# 7. Join Incremental Customers to Silver
# ============================================================

comparison_df = (
    incremental_dedup_df.alias("src")
    .join(
        silver_customers_df.alias("tgt"),

        F.col("src.customer_id")
        ==
        F.col("tgt.customer_id"),

        how="left"
    )
)


# ============================================================
# 8. Detect Business Attribute Changes
# ============================================================

change_condition = (

    ~F.col("src.first_name_clean").eqNullSafe(
        F.col("tgt.first_name")
    )

    |

    ~F.col("src.last_name_clean").eqNullSafe(
        F.col("tgt.last_name")
    )

    |

    ~F.col("src.email_clean").eqNullSafe(
        F.col("tgt.email")
    )

    |

    ~F.col("src.phone_number_clean").eqNullSafe(
        F.col("tgt.phone_number")
    )

    |

    ~F.col("src.date_of_birth_cast").eqNullSafe(
        F.col("tgt.date_of_birth")
    )

    |

    ~F.col("src.gender_clean").eqNullSafe(
        F.col("tgt.gender")
    )

    |

    ~F.col("src.signup_date_cast").eqNullSafe(
        F.col("tgt.signup_date")
    )

    |

    ~F.col("src.customer_status_clean").eqNullSafe(
        F.col("tgt.customer_status")
    )

    |

    ~F.col("src.loyalty_tier_clean").eqNullSafe(
        F.col("tgt.loyalty_tier")
    )

    |

    ~F.col("src.preferred_language_clean").eqNullSafe(
        F.col("tgt.preferred_language")
    )
)


# ============================================================
# 9. Classify INSERT / UPDATE / UNCHANGED
# ============================================================

classified_df = (
    comparison_df
    .withColumn(
        "cdc_action",

        F.when(
            F.col("tgt.customer_id").isNull(),
            F.lit("INSERT")
        )

        .when(
            change_condition,
            F.lit("UPDATE")
        )

        .otherwise(
            F.lit("UNCHANGED")
        )
    )
)


# ============================================================
# 10. Build Clean CDC Output
# ============================================================

customer_cdc_df = (
    classified_df
    .select(
        F.col("src.customer_id").alias("customer_id"),
        F.col("src.first_name_clean").alias("first_name"),
        F.col("src.last_name_clean").alias("last_name"),
        F.col("src.email_clean").alias("email"),
        F.col("src.phone_number_clean").alias("phone_number"),
        F.col("src.date_of_birth_cast").alias("date_of_birth"),
        F.col("src.gender_clean").alias("gender"),
        F.col("src.signup_date_cast").alias("signup_date"),
        F.col("src.customer_status_clean").alias(
            "customer_status"
        ),
        F.col("src.loyalty_tier_clean").alias(
            "loyalty_tier"
        ),
        F.col("src.preferred_language_clean").alias(
            "preferred_language"
        ),
        F.col("src.created_at_cast").alias("created_at"),
        F.col("src.updated_at_cast").alias("updated_at"),

        F.col("src._source_file").alias("_source_file"),
        F.col("src._source_system").alias("_source_system"),
        F.col("src._load_type").alias("_load_type"),
        F.col("src._ingestion_timestamp").alias(
            "_ingestion_timestamp"
        ),
        F.col("src._pipeline_run_id").alias(
        "_pipeline_run_id"
        ),

        F.col("cdc_action")
    )
)


# ============================================================
# 11. CDC Summary
# ============================================================

if incremental_count > 0:

    (
        customer_cdc_df
        .groupBy("cdc_action")
        .count()
        .orderBy("cdc_action")
        .show()
    )

else:

    print(
        "No incremental customer rows "
        "available for CDC classification."
    )


# ============================================================
# 12. Preview Classified Records
# ============================================================

display(
    customer_cdc_df.limit(50)
)

Processing Run ID: 411bb89d-645f-4b97-adda-9d20c8d53951
Incremental customer rows: 10
+----------+-----+
|cdc_action|count|
+----------+-----+
|    INSERT|   10|
+----------+-----+



customer_id,first_name,last_name,email,phone_number,date_of_birth,gender,signup_date,customer_status,loyalty_tier,preferred_language,created_at,updated_at,_source_file,_source_system,_load_type,_ingestion_timestamp,_pipeline_run_id,cdc_action
CINC000017,Customer7,Incremental,customer7@example.com,9810000006,1996-07-07,M,2026-09-30,ACTIVE,GOLD,TELUGU,2026-09-30T10:06:00.000Z,2026-09-30T10:07:00.000Z,dbfs:/Volumes/customer360_dev/staging/source_files/customers/incremental/customers_incremental_20261001.csv,CRM,INCREMENTAL,2026-10-01T10:29:12.885Z,411bb89d-645f-4b97-adda-9d20c8d53951,INSERT
CINC000018,Customer8,Incremental,customer8@example.com,9810000007,1997-08-08,F,2026-09-30,ACTIVE,PLATINUM,HINDI,2026-09-30T10:07:00.000Z,2026-09-30T10:08:00.000Z,dbfs:/Volumes/customer360_dev/staging/source_files/customers/incremental/customers_incremental_20261001.csv,CRM,INCREMENTAL,2026-10-01T10:29:12.885Z,411bb89d-645f-4b97-adda-9d20c8d53951,INSERT
CINC000011,Customer1,Incremental,customer1@example.com,9810000000,1990-01-01,M,2026-09-30,ACTIVE,BRONZE,ENGLISH,2026-09-30T10:00:00.000Z,2026-09-30T10:01:00.000Z,dbfs:/Volumes/customer360_dev/staging/source_files/customers/incremental/customers_incremental_20261001.csv,CRM,INCREMENTAL,2026-10-01T10:29:12.885Z,411bb89d-645f-4b97-adda-9d20c8d53951,INSERT
CINC000019,Customer9,Incremental,customer9@example.com,9810000008,1990-09-09,M,2026-09-30,ACTIVE,BRONZE,KANNADA,2026-09-30T10:08:00.000Z,2026-09-30T10:09:00.000Z,dbfs:/Volumes/customer360_dev/staging/source_files/customers/incremental/customers_incremental_20261001.csv,CRM,INCREMENTAL,2026-10-01T10:29:12.885Z,411bb89d-645f-4b97-adda-9d20c8d53951,INSERT
CINC000015,Customer5,Incremental,customer5@example.com,9810000004,1994-05-05,M,2026-09-30,ACTIVE,BRONZE,TAMIL,2026-09-30T10:04:00.000Z,2026-09-30T10:05:00.000Z,dbfs:/Volumes/customer360_dev/staging/source_files/customers/incremental/customers_incremental_20261001.csv,CRM,INCREMENTAL,2026-10-01T10:29:12.885Z,411bb89d-645f-4b97-adda-9d20c8d53951,INSERT
CINC000020,Customer10,Incremental,customer10@example.com,9810000009,1991-01-10,F,2026-09-30,ACTIVE,SILVER,TAMIL,2026-09-30T10:09:00.000Z,2026-09-30T10:10:00.000Z,dbfs:/Volumes/customer360_dev/staging/source_files/customers/incremental/customers_incremental_20261001.csv,CRM,INCREMENTAL,2026-10-01T10:29:12.885Z,411bb89d-645f-4b97-adda-9d20c8d53951,INSERT
CINC000016,Customer6,Incremental,customer6@example.com,9810000005,1995-06-06,F,2026-09-30,ACTIVE,SILVER,ENGLISH,2026-09-30T10:05:00.000Z,2026-09-30T10:06:00.000Z,dbfs:/Volumes/customer360_dev/staging/source_files/customers/incremental/customers_incremental_20261001.csv,CRM,INCREMENTAL,2026-10-01T10:29:12.885Z,411bb89d-645f-4b97-adda-9d20c8d53951,INSERT
CINC000014,Customer4,Incremental,customer4@example.com,9810000003,1993-04-04,F,2026-09-30,ACTIVE,PLATINUM,KANNADA,2026-09-30T10:03:00.000Z,2026-09-30T10:04:00.000Z,dbfs:/Volumes/customer360_dev/staging/source_files/customers/incremental/customers_incremental_20261001.csv,CRM,INCREMENTAL,2026-10-01T10:29:12.885Z,411bb89d-645f-4b97-adda-9d20c8d53951,INSERT
CINC000013,Customer3,Incremental,customer3@example.com,9810000002,1992-03-03,M,2026-09-30,ACTIVE,GOLD,HINDI,2026-09-30T10:02:00.000Z,2026-09-30T10:03:00.000Z,dbfs:/Volumes/customer360_dev/staging/source_files/customers/incremental/customers_incremental_20261001.csv,CRM,INCREMENTAL,2026-10-01T10:29:12.885Z,411bb89d-645f-4b97-adda-9d20c8d53951,INSERT
CINC000012,Customer2,Incremental,customer2@example.com,9810000001,1991-02-02,F,2026-09-30,ACTIVE,SILVER,TELUGU,2026-09-30T10:01:00.000Z,2026-09-30T10:02:00.000Z,dbfs:/Volumes/customer360_dev/staging/source_files/customers/incremental/customers_incremental_20261001.csv,CRM,INCREMENTAL,2026-10-01T10:29:12.885Z,411bb89d-645f-4b97-adda-9d20c8d53951,INSERT


In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

target_table = "customer360_dev.silver.customers"

silver_delta = DeltaTable.forName(
    spark,
    target_table
)

In [0]:
insert_count = (
    customer_cdc_df
    .filter(F.col("cdc_action") == "INSERT")
    .count()
)

update_count = (
    customer_cdc_df
    .filter(F.col("cdc_action") == "UPDATE")
    .count()
)

unchanged_count = (
    customer_cdc_df
    .filter(F.col("cdc_action") == "UNCHANGED")
    .count()
)

print(f"INSERT    : {insert_count:,}")
print(f"UPDATE    : {update_count:,}")
print(f"UNCHANGED : {unchanged_count:,}")

INSERT    : 10
UPDATE    : 0
UNCHANGED : 0


In [0]:
silver_before_count = (
    spark.table(target_table).count()
)

print(
    f"Silver Customers before merge: "
    f"{silver_before_count:,}"
)

Silver Customers before merge: 986,010


In [0]:
merge_source_df = (
    customer_cdc_df
    .filter(
        F.col("cdc_action").isin(
            "INSERT",
            "UPDATE"
        )
    )
)

# Write CDC work table BEFORE the silver MERGE to preserve correct INSERT/UPDATE classifications
(
    customer_cdc_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "customer360_dev.control.customer_cdc_work"
    )
)

Perform Delta MERGE

In [0]:
(
    silver_delta.alias("tgt")

    .merge(
        merge_source_df.alias("src"),
        "tgt.customer_id = src.customer_id"
    )

    .whenMatchedUpdate(
        condition="""
            src.cdc_action = 'UPDATE'
            AND (
                tgt.updated_at IS NULL
                OR src.updated_at >= tgt.updated_at
            )
        """,

        set={
            "first_name":
                "src.first_name",

            "last_name":
                "src.last_name",

            "email":
                "src.email",

            "phone_number":
                "src.phone_number",

            "date_of_birth":
                "src.date_of_birth",

            "gender":
                "src.gender",

            "signup_date":
                "src.signup_date",

            "customer_status":
                "src.customer_status",

            "loyalty_tier":
                "src.loyalty_tier",

            "preferred_language":
                "src.preferred_language",

            "created_at":
                "src.created_at",

            "updated_at":
                "src.updated_at",

            "_source_file":
                "src._source_file",

            "_source_system":
                "src._source_system",

            "_load_type":
                "src._load_type",

            "_ingestion_timestamp":
                "src._ingestion_timestamp",

            "_silver_processed_at":
                "current_timestamp()"
        }
    )

    .whenNotMatchedInsert(
        condition="""
            src.cdc_action = 'INSERT'
        """,

        values={
            "customer_id":
                "src.customer_id",

            "first_name":
                "src.first_name",

            "last_name":
                "src.last_name",

            "email":
                "src.email",

            "phone_number":
                "src.phone_number",

            "date_of_birth":
                "src.date_of_birth",

            "gender":
                "src.gender",

            "signup_date":
                "src.signup_date",

            "customer_status":
                "src.customer_status",

            "loyalty_tier":
                "src.loyalty_tier",

            "preferred_language":
                "src.preferred_language",

            "created_at":
                "src.created_at",

            "updated_at":
                "src.updated_at",

            "_source_file":
                "src._source_file",

            "_source_system":
                "src._source_system",

            "_load_type":
                "src._load_type",

            "_ingestion_timestamp":
                "src._ingestion_timestamp",

            "_silver_processed_at":
                "current_timestamp()"
        }
    )

    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
silver_after_count = (
    spark.table(target_table).count()
)

actual_growth = (
    silver_after_count
    - silver_before_count
)

print(
    f"Silver before : {silver_before_count:,}"
)

print(
    f"Silver after  : {silver_after_count:,}"
)

print(
    f"Actual growth : {actual_growth:,}"
)

print(
    f"Expected inserts : {insert_count:,}"
)

Silver before : 986,010
Silver after  : 986,020
Actual growth : 10
Expected inserts : 10


In [0]:
duplicate_customer_ids = (
    spark.table(target_table)
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate customer IDs in Silver:",
    duplicate_customer_ids
)

Duplicate customer IDs in Silver: 0


In [0]:
(
    spark.table(target_table)
    .filter(
        F.col("_load_type") == "INCREMENTAL"
    )
    .select(
        "customer_id",
        "email",
        "customer_status",
        "loyalty_tier",
        "updated_at",
        "_load_type",
        "_source_file"
    )
    .show(
        50,
        truncate=False
    )
)

+-----------+----------------------+---------------+------------+-------------------+-----------+-----------------------------------------------------------------------------------------------------------+
|customer_id|email                 |customer_status|loyalty_tier|updated_at         |_load_type |_source_file                                                                                               |
+-----------+----------------------+---------------+------------+-------------------+-----------+-----------------------------------------------------------------------------------------------------------+
|CINC000001 |customer1@example.com |ACTIVE         |BRONZE      |2026-09-30 10:01:00|INCREMENTAL|dbfs:/Volumes/customer360_dev/staging/source_files/customers/incremental/customers_incremental_20260930.csv|
|CINC000003 |customer3@example.com |ACTIVE         |GOLD        |2026-09-30 10:03:00|INCREMENTAL|dbfs:/Volumes/customer360_dev/staging/source_files/customers/incremental/custom

In [0]:
display(
    spark.table(
        "customer360_dev.control.customer_cdc_work"
    )
    .select(
        "customer_id",
        "loyalty_tier",
        "customer_status",
        "cdc_action",
        "_pipeline_run_id"
    )
    .limit(50)
)

customer_id,loyalty_tier,customer_status,cdc_action,_pipeline_run_id
CINC000019,BRONZE,ACTIVE,INSERT,411bb89d-645f-4b97-adda-9d20c8d53951
CINC000013,GOLD,ACTIVE,INSERT,411bb89d-645f-4b97-adda-9d20c8d53951
CINC000015,BRONZE,ACTIVE,INSERT,411bb89d-645f-4b97-adda-9d20c8d53951
CINC000016,SILVER,ACTIVE,INSERT,411bb89d-645f-4b97-adda-9d20c8d53951
CINC000018,PLATINUM,ACTIVE,INSERT,411bb89d-645f-4b97-adda-9d20c8d53951
CINC000020,SILVER,ACTIVE,INSERT,411bb89d-645f-4b97-adda-9d20c8d53951
CINC000011,BRONZE,ACTIVE,INSERT,411bb89d-645f-4b97-adda-9d20c8d53951
CINC000014,PLATINUM,ACTIVE,INSERT,411bb89d-645f-4b97-adda-9d20c8d53951
CINC000017,GOLD,ACTIVE,INSERT,411bb89d-645f-4b97-adda-9d20c8d53951
CINC000012,SILVER,ACTIVE,INSERT,411bb89d-645f-4b97-adda-9d20c8d53951
